In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, Ridge
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV


In [2]:
# Load the dataset and select only white wine data
dfw = pd.read_csv('datasets/winequality-white.csv', delimiter=';')
dfw.fillna(dfw.mean(), inplace=True)
X = dfw.loc[:, dfw.columns != 'quality']
y = dfw['quality']  # Target
dfw.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [3]:
from sklearn.preprocessing import StandardScaler


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#bias term to the features
X_train = np.c_[np.ones(X_train.shape[0]), X_train]
X_test = np.c_[np.ones(X_test.shape[0]), X_test]


In [4]:
class RidgeRegressionGD:
    def __init__(self, learning_rate=0.001, n_iterations=1000, alpha=0.1):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.alpha = alpha
        self.theta = None
    
    def fit(self, X, y):
        m, n = X.shape
        self.theta = np.zeros(n)
        

        for i in range(self.n_iterations):
            predictions = X.dot(self.theta)
            errors = predictions - y
            
            gradient = (1 / m) * (X.T.dot(errors)) + self.alpha * self.theta
            self.theta -= self.learning_rate * gradient
                
    
    def predict(self, X):
        return X.dot(self.theta)


# Train the model
ridge_gd = RidgeRegressionGD(learning_rate=0.001, n_iterations=1000, alpha=0.1)
ridge_gd.fit(X_train, y_train)

# Predict and evaluate
m_pred = ridge_gd.predict(X_test)
print("Final cost:", mean_squared_error(m_pred, y_test))


Final cost: 5.539713916460667


In [5]:
class RidgeRegressionSGD:
    def __init__(self, learning_rate=0.001, n_iterations=1000, alpha=0.1):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.alpha = alpha
        self.theta = None
    
    def fit(self, X, y):
        # Ensure X and y are NumPy arrays (they should be after scaling and train-test split)
        X = np.array(X)
        y = np.array(y)
        
        m, n = X.shape
        self.theta = np.zeros(n)
        
        for _ in range(self.n_iterations):
            # Create a random permutation of indices
            for i in np.random.permutation(m):
                xi = X[i]  # xi is the row from X
                yi = y[i]  # yi is the corresponding value from y
                error = np.dot(xi, self.theta) - yi
                gradient = xi * error + self.alpha * self.theta
                self.theta -= self.learning_rate * gradient
    
    def predict(self, X):
        return np.dot(X, self.theta)

# Train the model
ridge_gd = RidgeRegressionSGD(learning_rate=0.001, n_iterations=1000, alpha=0.1)
ridge_gd.fit(X_train, y_train)

# Predict and evaluate
m_pred = ridge_gd.predict(X_test)
print("Final cost:", mean_squared_error(m_pred, y_test))


Final cost: 0.6472871888510957


In [6]:
import time
from sklearn.metrics import mean_squared_error

learning_rates = [0.001, 0.003,0.06, 0.01, 0.03, 0.06, 0.1]

# For Ridge GD
opt_lr_gd = None
opt_mse_gd = None
convergence_time_gd = {}

print("Ridge GD:")
for lr in learning_rates:
    start_time_gd = time.time()
    model = RidgeRegressionGD(learning_rate=lr, alpha=0.1, n_iterations=1000)
    ridge_gd = model.fit(X_train, y_train)
    pred_gd = model.predict(X_test)
    mse_gd = mean_squared_error(y_test, pred_gd)
    convergence_time_gd[lr] = time.time() - start_time_gd
    
    print(f"Learning rate: {lr}; MSE: {mse_gd}; Time: {convergence_time_gd[lr]} seconds")
    
    # Track the optimal learning rate based on MSE
    if opt_lr_gd is None or mse_gd < opt_mse_gd:
        opt_lr_gd = lr
        opt_mse_gd = mse_gd

print(f"Ridge GD: Optimal learning rate: {opt_lr_gd}; MSE: {opt_mse_gd}")
print(f"Convergence time for GD: {convergence_time_gd[opt_lr_gd]} seconds")

# For Ridge SGD
opt_lr_sgd = None
opt_mse_sgd = None
convergence_time_sgd = {}

print("\nRidge SGD:")
for lr in learning_rates:
    start_time_sgd = time.time()
    model = RidgeRegressionSGD(learning_rate=lr, alpha=0.1, n_iterations=1000)
    ridge_sgd = model.fit(X_train, y_train)
    pred_sgd = model.predict(X_test)
    mse_sgd = mean_squared_error(y_test, pred_sgd)
    convergence_time_sgd[lr] = time.time() - start_time_sgd
    
    print(f"Learning rate: {lr}; MSE: {mse_sgd}; Time: {convergence_time_sgd[lr]} seconds")
    
    # Track the optimal learning rate based on MSE
    if opt_lr_sgd is None or mse_sgd < opt_mse_sgd:
        opt_lr_sgd = lr
        opt_mse_sgd = mse_sgd

print(f"Ridge SGD: Optimal learning rate: {opt_lr_sgd}; MSE: {opt_mse_sgd}")
print(f"Convergence time for SGD: {convergence_time_sgd[opt_lr_sgd]} seconds")


Ridge GD:
Learning rate: 0.001; MSE: 5.539713916460667; Time: 0.2893493175506592 seconds
Learning rate: 0.003; MSE: 0.8849917969657778; Time: 0.15964913368225098 seconds
Learning rate: 0.06; MSE: 0.658802219882365; Time: 0.15627551078796387 seconds
Learning rate: 0.01; MSE: 0.6598070207731823; Time: 0.3093414306640625 seconds
Learning rate: 0.03; MSE: 0.6588519089067109; Time: 0.3137083053588867 seconds
Learning rate: 0.06; MSE: 0.658802219882365; Time: 0.27665138244628906 seconds
Learning rate: 0.1; MSE: 0.6588010422671127; Time: 0.27457261085510254 seconds
Ridge GD: Optimal learning rate: 0.1; MSE: 0.6588010422671127
Convergence time for GD: 0.27457261085510254 seconds

Ridge SGD:
Learning rate: 0.001; MSE: 0.6535899552467331; Time: 23.21731925010681 seconds
Learning rate: 0.003; MSE: 0.6317247614212858; Time: 22.955113887786865 seconds
Learning rate: 0.06; MSE: 0.7476518167899296; Time: 22.999897241592407 seconds
Learning rate: 0.01; MSE: 0.6970754965566731; Time: 22.9963481426239 s

In [ ]:
class RidgeRegressionGD:
    def __init__(self, learning_rate=0.01, alpha=0.1, max_iter=1000):
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.max_iter = max_iter
        self.theta = None
        

    def _compute_gradient(self, X: np.ndarray, y: np.ndarray):
        """Compute the gradient for Ridge Regression."""
        m = len(y)
        predictions = X.dot(self.theta)
        errors = predictions - y
        return (1 / m) * X.T.dot(errors) + self.alpha * self.theta

    def fit(self, X: np.ndarray, y: np.ndarray):
        """Fit the Ridge Regression model using gradient descent."""
        n_features = X.shape[1]
        self.theta = np.zeros(n_features)  # Initialize coefficients

        for _ in range(self.max_iter):
            gradient = self._compute_gradient(X, y)
            update = self.learning_rate * gradient

            self.theta -= update
            

    def predict(self, X: np.ndarray):
        return X.dot(self.theta)


In [13]:
# Initialize and train the Ridge Regression model
model = RidgeRegressionGD(learning_rate=0.01, alpha=0.1, max_iter=1000)
model.fit(X_train, y_train)

# Predictions
predictions = model.predict(X_test)
print("Predicted values:", mean_squared_error(predictions, y_test))

Predicted values: 0.6598070207731823
